# FDC 설비 텔레메트리 스트림 노트북

설비 센서 판독값을 만들어 이 작업 영역의 **Eventhouse(KQL DB)** 에 적재합니다.
Fabric 스케줄러로 몇 분마다 돌리면 실제 팹처럼 시계열이 계속 쌓입니다.

## 세 번째 시스템

MES는 무엇을 만들었는지, QMS는 그것이 합격인지 압니다. FDC는 **설비가 그때
어떤 상태였는지**를 압니다. 셋은 서로 다른 시스템이고 공유하는 키는
`eqp_id` 와 `step_code` 뿐입니다.

FDC 판독값에는 `lot_id`, `product_code`, `defect_code`, `judgment` 가 **없습니다**.
센서는 지금 어떤 로트가 올라와 있는지 모르기 때문입니다. 그래서
"챔버 온도가 튄 그 시각에 어떤 로트가 그 설비에 있었나"를 알려면 MES에 물어야
합니다. 이것이 이 실습의 목표입니다.

## 이상은 MES에서 유도합니다

"고장 설비 목록"을 하드코딩하지 않습니다. MES `process_results` 의 설비별
불량률을 집계해, 불량률이 높은 설비일수록 크게 이탈시키고 불량코드가
지목하는 센서에만 이탈을 싣습니다. Eventhouse에서 찾아낸 이상 설비가 MES
실적과 맞아떨어지는 이유입니다.

## 실행 순서

1. 작업 영역에 **Eventhouse** 를 만들고 KQL 데이터베이스를 하나 둡니다.
2. KQL 데이터베이스 페이지 오른쪽 위 **Query URI** 를 복사합니다.
3. 아래 파라미터 셀에 그 URI와 데이터베이스 이름, MES API 키를 넣습니다.
4. 전체 실행합니다.

Query URI는 비밀값이 아닙니다. 인증은 `mssparkutils` 가 실행자 신원으로
토큰을 발급해 처리하므로 복사해 둘 키가 없습니다.

## 두 번째 실행부터

첫 실행은 MES 공정이력이 걸쳐 있는 구간 전체를 백필합니다. 약 65시간이고
10만 행 안팎입니다. 이후 실행은 이미 적재된 마지막 시각(watermark)부터
지금까지만 채웁니다. 값이 (설비, 센서, 타임스탬프)만으로 정해지므로 몇 번을
다시 돌려도 같은 시각에는 같은 값이 들어갑니다.

설비가 돌고 있을 때만 공정 센서 6종을 30초 간격으로 내보냅니다. 멈춰 있는
동안에는 주변 온도·습도 2종만 5분 간격으로 남습니다. `run_status` 컬럼으로
구분할 수 있습니다.


In [ ]:
# Fabric 파이프라인이나 스케줄러에서 이 셀의 값을 덮어쓸 수 있습니다.

# KQL 데이터베이스 페이지 오른쪽 위의 Query URI 입니다. 비밀값이 아닙니다.
# 예: "https://trd-abcdefg.z9.kusto.fabric.microsoft.com"
KUSTO_URI = ""
KUSTO_DATABASE = ""

# MES 인증 키. 이 작업 영역은 공유될 수 있고 노트북은 자동 저장되니 실행 후 지우세요.
MES_BASE_URL = "https://mock-mes.greenrock-bb44c93a.koreacentral.azurecontainerapps.io"
MES_API_KEY = ""


In [ ]:
"""Mock MES 접속 프로브. MCP 채널만 쓴다.

FDC 텔레메트리가 MES에서 필요로 하는 사실은 두 가지뿐이다.

- 설비 목록: 어떤 설비에서 센서를 읽을 것인가
- 설비별 불량 실적: 어느 설비에 이상을 주입할 것인가

둘 다 `list_process_results` 와 `get_process_route` 로 얻는다. REST 채널
(`/api/products` 등)과 제품·자재·BOM·로트는 쓰지 않으므로 가져오지 않는다.

표준 라이브러리만 사용한다. Fabric 노트북에 그대로 인라인되기 때문에
추가 패키지 설치가 있어선 안 된다.
"""

from __future__ import annotations

import json
import urllib.request
from dataclasses import dataclass, field
from typing import Any

MES_BASE_URL = "https://mock-mes.greenrock-bb44c93a.koreacentral.azurecontainerapps.io"

# 쓰기 툴은 호출하지 않는다. 이 노트북은 MES를 읽기만 한다.
_WRITE_TOOLS = frozenset({"start_lot", "register_process_result"})


class MesApiKeyMissing(ValueError):
    """MES_API_KEY 가 비어 있을 때."""


def parse_mcp_body(raw: str) -> dict | None:
    """MCP 응답 본문을 파싱한다.

    이 서버는 Accept 헤더에 text/event-stream 이 있으면 SSE 로 답한다.
    본문이 'event: message' 로 시작하므로 'data:' 접두 검사만으로는 SSE를
    인식할 수 없다. 순수 JSON을 먼저 시도하고 실패하면 data 행을 모은다.
    """
    body = raw.strip()
    if not body:
        return None
    try:
        return json.loads(body)
    except json.JSONDecodeError:
        pass
    chunks = [
        line[len("data:"):].strip()
        for line in body.splitlines()
        if line.strip().startswith("data:")
    ]
    if not chunks:
        raise ValueError(f"MCP 응답을 해석할 수 없습니다: {body[:200]!r}")
    return json.loads("".join(chunks))


def derive_equipment(process_results: list[dict], route: list[dict]) -> list[dict]:
    """공정이력의 eqp_id 고유값에서 설비 목록을 만든다.

    MES는 설비 마스터를 REST에도 MCP에도 노출하지 않는다. eqp_id 가 비어 있는
    공정이력이 존재하므로(실측 91건 중 7건) 그 행은 건너뛴다.
    """
    eqp_type_by_step = {s["step_code"]: s.get("eqp_type") for s in route}
    first_step: dict[str, str] = {}
    for row in process_results:
        eqp_id = row.get("eqp_id")
        if eqp_id and eqp_id not in first_step:
            first_step[eqp_id] = row["step_code"]
    return [
        {
            "eqp_id": eqp_id,
            "eqp_type": eqp_type_by_step.get(first_step[eqp_id]),
            "step_code": first_step[eqp_id],
        }
        for eqp_id in sorted(first_step)
    ]


@dataclass(frozen=True)
class MesFacts:
    """FDC 생성기의 유일한 MES 입력. 네트워크 계층과 생성 계층의 경계다."""

    process_results: list[dict]
    route: list[dict]
    equipment: list[dict] = field(default_factory=list)

    def __post_init__(self) -> None:
        if not self.equipment:
            object.__setattr__(
                self, "equipment", derive_equipment(self.process_results, self.route)
            )

    @classmethod
    def from_dict(cls, payload: dict) -> MesFacts:
        return cls(
            process_results=payload["process_results"],
            route=payload["route"],
            equipment=payload.get("equipment") or [],
        )

    def to_dict(self) -> dict:
        return {
            "process_results": self.process_results,
            "route": self.route,
            "equipment": self.equipment,
        }


class MesProbe:
    def __init__(self, base_url: str = MES_BASE_URL, api_key: str = "", timeout: int = 60):
        if not api_key:
            raise MesApiKeyMissing(
                "MES_API_KEY가 비어 있습니다. 노트북 파라미터 셀에 키를 넣으세요."
            )
        self.base_url = base_url.rstrip("/")
        self.api_key = api_key
        self.timeout = timeout
        self._rpc_id = 0
        self._initialized = False

    def _post(self, payload: dict) -> str:
        request = urllib.request.Request(
            self.base_url + "/mcp",
            data=json.dumps(payload).encode("utf-8"),
            headers={
                "Content-Type": "application/json",
                "Accept": "application/json, text/event-stream",
                "X-API-Key": self.api_key,
            },
            method="POST",
        )
        with urllib.request.urlopen(request, timeout=self.timeout) as response:
            return response.read().decode("utf-8")

    def _rpc(self, method: str, params: dict, notify: bool = False) -> dict | None:
        payload: dict[str, Any] = {"jsonrpc": "2.0", "method": method, "params": params}
        if not notify:
            self._rpc_id += 1
            payload["id"] = self._rpc_id
        return parse_mcp_body(self._post(payload))

    def _handshake(self) -> None:
        if self._initialized:
            return
        self._rpc(
            "initialize",
            {
                "protocolVersion": "2024-11-05",
                "capabilities": {},
                "clientInfo": {"name": "fdc-telemetry", "version": "1.0"},
            },
        )
        self._rpc("notifications/initialized", {}, notify=True)
        self._initialized = True

    def mcp_call(self, tool: str, args: dict | None = None) -> Any:
        """읽기 전용 MCP 툴 호출. 결과를 그대로 돌려준다."""
        if tool in _WRITE_TOOLS:
            raise ValueError(f"쓰기 툴 호출 금지: {tool}")
        self._handshake()
        response = self._rpc("tools/call", {"name": tool, "arguments": args or {}})
        if response is None:
            raise RuntimeError(f"MCP 툴 {tool} 응답이 비어 있습니다.")
        if "error" in response:
            raise RuntimeError(f"MCP 툴 {tool} 오류: {response['error']}")
        if "result" not in response:
            raise RuntimeError(f"MCP 툴 {tool} 응답에 result가 없습니다: {response}")
        result = response["result"]
        if result.get("isError"):
            raise RuntimeError(f"MCP 툴 {tool} 실패: {result.get('content')}")
        structured = result.get("structuredContent")
        if not isinstance(structured, dict) or "result" not in structured:
            raise RuntimeError(f"MCP 툴 {tool} 응답 형식이 예상과 다릅니다: {result}")
        return structured["result"]

    def fetch_facts(self) -> MesFacts:
        process_results = self.mcp_call("list_process_results", {"limit": 500})
        route = self.mcp_call("get_process_route")
        return MesFacts(process_results=process_results, route=route)


In [ ]:
"""MES 공정이력을 설비별 런 구간으로 바꾼다. 전부 순수 함수다.

판독값을 가동과 유휴로 나누고 이상을 어느 구간에 실을지 정하려면 "이 시각에
이 설비가 무엇을 하고 있었나"를 알아야 한다. 그 정보는 MES 공정이력의
in_time/out_time 에만 있다.

구간은 [start, end) 반열림이다. 한 설비의 두 런이 경계에서 맞닿아도 한 시각이
양쪽에 속하지 않는다.

Run.lot_id 는 이상 배치 계산에만 쓰고 판독값에는 싣지 않는다. FDC 가 로트를
들고 있으면 실습자가 Eventhouse 하나로 답을 내버려서 MES 에 물을 이유가
사라진다. 자세한 근거는 계획서 "설계 결정" 절에 있다.
"""

from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timezone


@dataclass(frozen=True)
class Run:
    eqp_id: str
    lot_id: str
    step_code: str
    start: datetime
    end: datetime
    defect_code: str | None


def parse_ts(value: str) -> datetime:
    """MES 시각 문자열을 tz-aware UTC datetime 으로."""
    text = value.strip()
    if text.endswith(("Z", "z")):
        text = text[:-1] + "+00:00"
    parsed = datetime.fromisoformat(text)
    if parsed.tzinfo is None:
        parsed = parsed.replace(tzinfo=timezone.utc)
    return parsed.astimezone(timezone.utc)


def runs_by_equipment(facts) -> dict[str, list[Run]]:
    """설비별 런 구간. start 오름차순.

    eqp_id 가 없는 공정이력은 버린다. METRO(계측)는 설비를 배정받지 않으므로
    센서 데이터가 존재할 수 없다. 모든 공정에 FDC 가 붙어 있지는 않다는 것
    자체가 현실적인 교육 소재다.
    """
    grouped: dict[str, list[Run]] = {}
    for row in facts.process_results:
        eqp_id = row.get("eqp_id")
        if not eqp_id:
            continue
        grouped.setdefault(eqp_id, []).append(
            Run(
                eqp_id=eqp_id,
                lot_id=row["lot_id"],
                step_code=row["step_code"],
                start=parse_ts(row["in_time"]),
                end=parse_ts(row["out_time"]),
                defect_code=row.get("defect_code") or None,
            )
        )
    for runs in grouped.values():
        runs.sort(key=lambda r: r.start)
    return grouped


def run_at(runs: list[Run], moment: datetime) -> Run | None:
    """이 시각에 돌고 있던 런. 없으면 None.

    선형 탐색이다. 설비당 런이 20건 미만이라 이분 탐색을 넣을 이유가 없다.
    start 오름차순이므로 시작이 moment 를 지나면 더 볼 필요가 없다.
    """
    for run in runs:
        if run.start <= moment < run.end:
            return run
        if run.start > moment:
            break
    return None


def span(facts) -> tuple[datetime, datetime]:
    """공정이력 전체가 걸쳐 있는 구간.

    설비가 없는 행도 포함한다. 생성 범위의 시작점을 정하는 용도라
    "MES 가 아는 가장 이른 시각"이어야 한다.
    """
    rows = facts.process_results
    if not rows:
        raise ValueError("공정이력이 비어 있어 생성 구간을 정할 수 없습니다.")
    return (
        min(parse_ts(r["in_time"]) for r in rows),
        max(parse_ts(r["out_time"]) for r in rows),
    )


In [ ]:
"""설비 유형별 센서 정의.

설비 유형 7종이 각각 공통 센서 2종과 고유 센서 4종을 갖는다. 합계 42종이며
이것이 `fdc_sensor_spec` 테이블이 된다.

공통 센서(주변 온도·습도)를 모든 유형에 두는 이유는 두 가지다. 클린룸 환경은
설비 종류와 무관하게 측정되고, 전 설비에 걸친 동일 계열이 있어야 KQL
`make-series` 로 설비 간 비교 실습이 가능하다.

`base` 는 반드시 정상범위의 중심이어야 한다. 이상 주입(fdc_anomaly)이 중심
대칭을 전제로 이탈 폭을 계산하기 때문이다. 테스트가 이 불변식을 강제한다.
"""

from __future__ import annotations

from dataclasses import dataclass

SAMPLE_INTERVAL_SEC = 30


@dataclass(frozen=True)
class SensorDef:
    sensor_code: str
    sensor_name_ko: str
    unit: str
    base: float
    diurnal_amp: float
    sigma: float
    normal_min: float
    normal_max: float
    alarm_min: float
    alarm_max: float


# 모든 설비 유형이 갖는다. 클린룸 환경 계측.
COMMON_SENSORS: tuple[SensorDef, ...] = (
    SensorDef("AMBIENT_TEMP", "주변 온도", "degC", 22.0, 0.6, 0.12, 21.0, 23.0, 20.0, 24.0),
    SensorDef("AMBIENT_HUMIDITY", "주변 습도", "%", 45.0, 2.5, 0.6, 40.0, 50.0, 35.0, 55.0),
)

TYPE_SENSORS: dict[str, tuple[SensorDef, ...]] = {
    "Furnace": (
        SensorDef("CHAMBER_TEMP", "챔버 온도", "degC", 1050.0, 1.5, 1.2, 1040.0, 1060.0, 1030.0, 1070.0),
        SensorDef("RAMP_RATE", "승온 속도", "degC/min", 8.0, 0.1, 0.08, 7.5, 8.5, 7.0, 9.0),
        SensorDef("O2_CONC", "산소 농도", "ppm", 120.0, 3.0, 2.0, 100.0, 140.0, 80.0, 160.0),
        SensorDef("N2_FLOW", "질소 유량", "sccm", 2000.0, 15.0, 8.0, 1950.0, 2050.0, 1900.0, 2100.0),
    ),
    "Scanner": (
        SensorDef("STAGE_TEMP", "스테이지 온도", "degC", 23.0, 0.03, 0.02, 22.9, 23.1, 22.8, 23.2),
        SensorDef("FOCUS_OFFSET", "포커스 오프셋", "nm", 0.0, 2.0, 1.5, -15.0, 15.0, -25.0, 25.0),
        SensorDef("ILLUM_DOSE", "노광 도즈", "mJ/cm2", 30.0, 0.15, 0.1, 29.4, 30.6, 29.0, 31.0),
        SensorDef("RETICLE_TEMP", "레티클 온도", "degC", 23.0, 0.02, 0.012, 22.95, 23.05, 22.9, 23.1),
    ),
    "Etcher": (
        SensorDef("RF_POWER", "RF 파워", "W", 1500.0, 8.0, 6.0, 1450.0, 1550.0, 1400.0, 1600.0),
        SensorDef("CHAMBER_PRESSURE", "챔버 압력", "mTorr", 45.0, 0.6, 0.5, 42.0, 48.0, 40.0, 50.0),
        SensorDef("GAS_FLOW", "공정가스 유량", "sccm", 180.0, 2.0, 1.5, 172.0, 188.0, 165.0, 195.0),
        SensorDef("CHAMBER_TEMP", "챔버 온도", "degC", 65.0, 0.8, 0.5, 62.0, 68.0, 60.0, 70.0),
    ),
    "Implanter": (
        SensorDef("BEAM_CURRENT", "빔 전류", "uA", 500.0, 5.0, 4.0, 480.0, 520.0, 460.0, 540.0),
        SensorDef("BEAM_ENERGY", "빔 에너지", "keV", 80.0, 0.4, 0.3, 78.0, 82.0, 76.0, 84.0),
        SensorDef("VACUUM", "진공도", "uTorr", 2.0, 0.08, 0.06, 1.5, 2.5, 1.0, 3.5),
        SensorDef("SRC_TEMP", "이온소스 온도", "degC", 320.0, 3.0, 2.0, 305.0, 335.0, 290.0, 350.0),
    ),
    "CVD": (
        SensorDef("CHAMBER_TEMP", "챔버 온도", "degC", 420.0, 2.0, 1.5, 410.0, 430.0, 400.0, 440.0),
        SensorDef("CHAMBER_PRESSURE", "챔버 압력", "Torr", 5.0, 0.08, 0.06, 4.6, 5.4, 4.2, 5.8),
        SensorDef("PRECURSOR_FLOW", "전구체 유량", "sccm", 250.0, 2.5, 2.0, 240.0, 260.0, 230.0, 270.0),
        SensorDef("DEP_RATE", "증착 속도", "nm/min", 12.0, 0.15, 0.1, 11.4, 12.6, 11.0, 13.0),
    ),
    "Polisher": (
        SensorDef("PAD_PRESSURE", "패드 압력", "kPa", 35.0, 0.3, 0.25, 33.0, 37.0, 31.0, 39.0),
        SensorDef("SLURRY_FLOW", "슬러리 유량", "ml/min", 200.0, 2.0, 1.6, 190.0, 210.0, 180.0, 220.0),
        SensorDef("MOTOR_CURRENT", "모터 전류", "A", 18.0, 0.25, 0.2, 17.0, 19.0, 16.0, 20.0),
        SensorDef("PAD_TEMP", "패드 온도", "degC", 42.0, 0.6, 0.45, 39.0, 45.0, 37.0, 47.0),
    ),
    "Prober": (
        SensorDef("CHUCK_TEMP", "척 온도", "degC", 25.0, 0.1, 0.05, 24.7, 25.3, 24.5, 25.5),
        SensorDef("CONTACT_RES", "접촉 저항", "mohm", 50.0, 1.0, 0.8, 45.0, 55.0, 40.0, 60.0),
        SensorDef("PROBE_FORCE", "프로브 압력", "mN", 30.0, 0.3, 0.25, 28.0, 32.0, 26.0, 34.0),
        SensorDef("TOUCHDOWN_CNT", "터치다운 횟수", "cnt", 1200.0, 40.0, 25.0, 1000.0, 1400.0, 900.0, 1500.0),
    ),
}

EQP_TYPES: tuple[str, ...] = tuple(sorted(TYPE_SENSORS))


def sensors_for(eqp_type: str) -> tuple[SensorDef, ...]:
    """공통 2종 + 유형 고유 4종. 미지의 유형이면 공통 2종만 돌려준다."""
    return COMMON_SENSORS + TYPE_SENSORS.get(eqp_type, ())



def idle_sensors() -> tuple[SensorDef, ...]:
    """설비가 멈춰 있을 때도 의미가 있는 센서.

    챔버 압력이나 RF 파워는 멈춘 설비에서 측정 자체가 무의미하다. 그렇다고
    0 을 내보내면 RF_POWER 의 alarm_min 이 1400 이라 유휴 내내 경보가 된다.
    클린룸 주변 온도·습도는 설비 가동과 무관하게 계속 측정된다.
    """
    return COMMON_SENSORS

def sensor_by_code(eqp_type: str, sensor_code: str) -> SensorDef:
    for sensor in sensors_for(eqp_type):
        if sensor.sensor_code == sensor_code:
            return sensor
    raise KeyError(f"{eqp_type} 에 {sensor_code} 센서가 없습니다.")


def build_sensor_spec_rows() -> list[dict]:
    """fdc_sensor_spec 42행. 키는 (eqp_type, sensor_code)."""
    return [
        {
            "sensor_code": sensor.sensor_code,
            "sensor_name_ko": sensor.sensor_name_ko,
            "eqp_type": eqp_type,
            "unit": sensor.unit,
            "normal_min": sensor.normal_min,
            "normal_max": sensor.normal_max,
            "alarm_min": sensor.alarm_min,
            "alarm_max": sensor.alarm_max,
            "sample_interval_sec": SAMPLE_INTERVAL_SEC,
            "is_active": True,
        }
        for eqp_type in EQP_TYPES
        for sensor in sensors_for(eqp_type)
    ]


In [ ]:
"""MES 불량 실적에서 센서 이상을 유도한다.

이 모듈이 FDC와 MES를 잇는 지점이다. 텔레메트리를 난수로만 만들면 실습자가
"어느 설비가 이상한가"를 Eventhouse 안에서 찾아낸 뒤 MES에 물어봐도 답이
맞아떨어지지 않는다. 그래서 이탈의 크기와 대상 센서를 MES 실적에서 끌어온다.

- 얼마나: 설비의 불량률이 높을수록 이탈 진폭이 크다
- 어디에: 불량코드가 지목하는 센서 중 **런마다 하나**에만 이탈이 실린다
- 언제: 불량이 난 그 런의 `[in_time, out_time)` 구간에만 실린다

결과적으로 "CHAMBER_TEMP 가 튀는 설비"를 Eventhouse에서 찾으면 그 설비가
MES에서 실제로 불량이 많은 설비다. 나아가 **경보가 뜬 시각**을 MES에 물으면
그때 돌던 로트가 나온다. 두 시스템을 교차 질의할 이유가 생긴다.

FDC 는 로트를 모른다 — 판독값에 `lot_id` 를 남기지 않는다. 로트는 이 모듈이
이상을 어디에 실을지 정하는 계산에만 쓴다.
"""

from __future__ import annotations

import hashlib
from datetime import datetime
from dataclasses import dataclass


# 이탈 진폭. 정상범위 반폭을 1.0 으로 보는 단위다.
# 경보 임계는 정상 반폭의 1.5~3.0 배에 있으므로, 불량률 0 인 설비는 1.0 으로
# 경고에도 못 닿고 불량률 100% 인 설비는 3.4 로 확실히 경보를 낸다.
#
# 상한을 이보다 낮추면 안 되는 이유: 실제 픽스처에서 가장 깨끗한 설비의
# 불량률이 0 이 아니라 0.167 이다. MIN 을 키우면 그 설비까지 경보에 닿아
# test_lowest_severity_stays_below_min_alarm_ratio 가 깨진다.
#
# 이 여유는 구조적으로 얇다. 진폭이 설비 자신의 불량률만의 함수이므로 경계는
# 불량률 축 위의 한 점인데, 가장 깨끗한 두 설비의 간격이 0.015 밖에 안 된다
# (IMPL01 0.167 · PHOT01 0.182). 30개 조합을 실측해 본 결과 "7대 경보 + 최저
# 설비 침묵" 을 지키며 얻을 수 있는 최대 여유가 0.056 이었다. 상수 선택 탓이
# 아니라 데이터 구조 탓이므로, 여유를 늘리려면 여기가 아니라 MES 불량률
# 분포를 손봐야 한다.
EXCURSION_MIN = 1.0
EXCURSION_MAX = 3.4


# 불량코드가 지목하는 센서. 물리적 인과가 성립하는 것만 넣는다(설계 스펙 7.2).
DEFECT_SENSOR_HINT: dict[str, tuple[str, ...]] = {
    # 온도 급변 시 챔버 박리물이 생긴다
    "Particle": ("CHAMBER_TEMP", "AMBIENT_HUMIDITY"),
    # 기계적 접촉 과다
    "Scratch": ("PAD_PRESSURE", "MOTOR_CURRENT", "PROBE_FORCE"),
    # 열팽창에 의한 정렬 오차
    "Overlay": ("AMBIENT_TEMP", "STAGE_TEMP", "RETICLE_TEMP"),
    # 반응 가스 부족
    "Etch-Residue": ("GAS_FLOW", "PRECURSOR_FLOW", "RF_POWER"),
    # 습도 상승과 진공도 저하
    "Contamination": ("AMBIENT_HUMIDITY", "VACUUM"),
    # 노광·식각 조건 이탈
    "CD-OOS": ("FOCUS_OFFSET", "RF_POWER", "ILLUM_DOSE"),
}

# 지목 센서가 그 설비 유형에 하나도 없을 때 대신 쓴다. 모든 유형이 갖는 센서여야 한다.
FALLBACK_SENSOR = "AMBIENT_TEMP"


def seed(*parts: object) -> int:
    """결정적 시드. [0, 2**32) 정수.

    내장 `hash()` 를 쓰면 안 된다. 파이썬은 문자열 해시에 프로세스마다 다른
    난수를 섞으므로(PYTHONHASHSEED) 같은 입력이 실행마다 다른 값을 낸다.
    이 노트북은 스케줄 실행마다 새 프로세스로 뜨며 백필과 라이브가 같은
    타임스탬프에 같은 값을 내야 한다.

    crc32 도 프로세스 고정이지만 GF(2) 위의 선형 함수라 입력이 몇 비트만
    다르면 출력 비트가 함께 움직인다. 실제로 설비명만 바꾼 48개 조합에서
    최하위 비트가 한쪽으로 몰려 이탈 방향이 거의 같은 쪽으로 쏠렸다.
    sha256 은 비선형이라 이런 뭉침이 없다. 여기서 sha256 은 보안 용도가
    아니라 결정적 혼합기로만 쓴다.
    """
    digest = hashlib.sha256("|".join(str(p) for p in parts).encode("utf-8")).digest()
    return int.from_bytes(digest[:4], "big")


def unit_from_seed(*parts: object) -> float:
    """시드를 [0, 1) 실수로. 난수 발생기 대신 쓴다."""
    return seed(*parts) / 2**32


@dataclass(frozen=True)
class EquipmentProfile:
    eqp_id: str
    eqp_type: str
    step_code: str
    total_runs: int
    defect_runs: int
    defect_codes: dict[str, int]
    severity: float  # 그 설비 자신의 불량률. defect_rate 와 같은 값이다

    @property
    def defect_rate(self) -> float:
        return self.defect_runs / self.total_runs if self.total_runs else 0.0


def build_profiles(facts) -> dict[str, EquipmentProfile]:
    """설비별 불량 프로파일.

    판정은 `defect_code` 유무로 본다. `judgment` 컬럼은 91건 전부 null 이라
    쓸 수 없고, `result` 는 Pass/Fail/Rework 3값이지만 불량코드가 붙은 행이
    Pass 로 남아 있는 경우가 있어 불량코드를 신호로 삼는다.
    """
    eqp_type_by_id = {e["eqp_id"]: e["eqp_type"] for e in facts.equipment}
    step_by_id = {e["eqp_id"]: e["step_code"] for e in facts.equipment}

    totals: dict[str, int] = {}
    defects: dict[str, int] = {}
    codes: dict[str, dict[str, int]] = {}
    for row in facts.process_results:
        eqp_id = row.get("eqp_id")
        if not eqp_id:
            continue
        totals[eqp_id] = totals.get(eqp_id, 0) + 1
        codes.setdefault(eqp_id, {})
        code = row.get("defect_code")
        if code:
            defects[eqp_id] = defects.get(eqp_id, 0) + 1
            codes[eqp_id][code] = codes[eqp_id].get(code, 0) + 1

    rates = {e: defects.get(e, 0) / totals[e] for e in totals}

    return {
        eqp_id: EquipmentProfile(
            eqp_id=eqp_id,
            eqp_type=eqp_type_by_id.get(eqp_id, ""),
            step_code=step_by_id.get(eqp_id, ""),
            total_runs=totals[eqp_id],
            defect_runs=defects.get(eqp_id, 0),
            defect_codes=dict(sorted(codes[eqp_id].items())),
            # 설비 자신의 불량률만 쓴다. 전체 설비의 min/max 로 정규화하면
            # 어느 설비 실적이 하나만 바뀌어도 나머지 일곱 대의 진폭이 전부
            # 흔들린다. Mock MES 는 쓰기 툴을 노출하고 20명이 한 인스턴스를
            # 공유하므로, 한 사람이 공정 실적을 등록하면 그 뒤에 백필한
            # 사람의 과거 판독값이 앞사람과 달라진다.
            severity=rates[eqp_id],
        )
        for eqp_id in sorted(totals)
    }


def fallback_sensor(eqp_type: str) -> str | None:
    """지목표가 이 설비에 없는 센서만 가리킬 때 대신 고를 센서.

    Mock MES 는 불량코드를 공정 단계와 무관하게 붙인다. 실제로 EQP-IMPL01
    (Implanter)에 Scratch 불량이 달려 있는데 Scratch 가 가리키는 센서는
    Implanter 에 하나도 없다. 그대로 두면 불량이 신호를 만들지 못한다.

    모든 설비 유형이 갖는 AMBIENT_TEMP 로 넘긴다. 클린룸 열관리 실패는
    실제로 여러 불량의 공통 원인이므로 물리적으로도 말이 된다.
    """
    available = {s.sensor_code for s in sensors_for(eqp_type)}
    return FALLBACK_SENSOR if FALLBACK_SENSOR in available else None


def hinted_sensors(profile: EquipmentProfile) -> dict[str, float]:
    """이탈을 실을 센서와 그 지목 비중. 합은 1.0.

    설비의 불량코드가 가리키는 센서 중 그 설비 유형에 실제로 달려 있는
    것만 남긴다. Furnace 에 RF_POWER 는 없으므로 Etch-Residue 불량이 있어도
    RF_POWER 에는 이탈을 실을 수 없다. 남는 게 없으면 폴백으로 넘긴다.
    """
    available = {s.sensor_code for s in sensors_for(profile.eqp_type)}
    weights: dict[str, float] = {}
    for code, count in profile.defect_codes.items():
        targets = [s for s in DEFECT_SENSOR_HINT.get(code, ()) if s in available]
        if not targets:
            chosen = fallback_sensor(profile.eqp_type)
            if chosen is None:
                continue
            targets = [chosen]
        for sensor_code in targets:
            weights[sensor_code] = weights.get(sensor_code, 0.0) + count / len(targets)
    total = sum(weights.values())
    return {k: v / total for k, v in sorted(weights.items())} if total else {}


def excursion_amplitude(profile: EquipmentProfile) -> float:
    """설비의 이탈 진폭 상한. 불량률이 높을수록 크다."""
    return EXCURSION_MIN + (EXCURSION_MAX - EXCURSION_MIN) * profile.severity


def candidate_sensors(defect_code: str | None, eqp_type: str) -> tuple[str, ...]:
    """이 불량이 지목하는 센서 중 그 설비에 실제로 달린 것.

    Mock MES 는 불량코드를 공정과 무관하게 붙인다. Implanter 에 Scratch 가
    달리면 지목 센서가 하나도 없고, 그때는 모든 유형이 갖는 AMBIENT_TEMP 로
    넘긴다. 클린룸 열관리 실패는 실제로 여러 불량의 공통 원인이다.
    """
    if not defect_code:
        return ()
    available = {s.sensor_code for s in sensors_for(eqp_type)}
    hinted = tuple(
        s for s in DEFECT_SENSOR_HINT.get(defect_code, ()) if s in available
    )
    if hinted:
        return hinted
    chosen = fallback_sensor(eqp_type)
    return (chosen,) if chosen else ()


def run_sensor(run, eqp_type: str) -> str | None:
    """이 런에서 실제로 흐르는 센서 하나. 불량이 없으면 None.

    후보 전부에 이탈을 나눠 실으면 진폭이 희석돼 어느 쪽도 경보에 못 닿는다.
    실제 공정 이탈도 보통 파라미터 하나가 흐르지 여러 개가 동시에 흐르지
    않는다. 런 식별자로 고르므로 같은 설비·같은 불량이라도 런마다 다른
    센서가 걸려 실습 소재가 다양해진다.
    """
    candidates = candidate_sensors(run.defect_code, eqp_type)
    if not candidates:
        return None
    index = seed(run.lot_id, run.step_code, run.eqp_id, run.defect_code)
    return candidates[index % len(candidates)]


def run_excursion(
    run, sensor_code: str, moment: datetime, profile: EquipmentProfile
) -> float:
    """런 구간 안의 이탈량. 정상범위 반폭이 1.0 인 단위.

    런이 진행될수록 커진다(progress 의 제곱). 공정이 서서히 이탈하다 끝에서
    불량으로 잡히는 모습이라 실습자에게 설명하기 쉽고, 런 앞부분이 잠잠해서
    경보가 끊이지 않는 일도 없다.
    """
    if not run.start <= moment < run.end:
        return 0.0
    if sensor_code != run_sensor(run, profile.eqp_type):
        return 0.0
    length = (run.end - run.start).total_seconds()
    if length <= 0:
        return 0.0
    progress = (moment - run.start).total_seconds() / length
    return excursion_sign(run.eqp_id, sensor_code) * excursion_amplitude(profile) * progress**2


def excursion_sign(eqp_id: str, sensor_code: str) -> int:
    """이탈 방향. 설비·센서마다 고정이며 위아래가 섞이게 한다."""
    return 1 if seed(eqp_id, sensor_code, "sign") % 2 == 0 else -1


In [ ]:
"""판독값 생성. MES 스냅샷을 고정하면 전부 순수 함수다.

이 모듈의 모든 함수는 주어진 MES 스냅샷 아래에서 같은 (설비, 센서,
타임스탬프)에 대해 언제 어디서 불러도 같은 값을 낸다. 순차 상태도, 실행
시각 의존도 없다.

그래야 하는 이유는 재실행 때문이다. 노트북은 실행마다 새 프로세스로 돌고,
장애로 걸렀던 구간을 나중에 백필한다. 값이 실행 시점에 좌우되면 백필한
구간과 정상 적재한 구간이 이어지지 않아 시계열에 계단이 생긴다.

MES 스냅샷이 입력인 것은 설계 목표다. "불량률이 높은 설비일수록 크게
이탈한다"는 관계가 이 데이터셋의 존재 이유이므로 실적을 없앨 수 없다.
다만 한 설비의 값은 **그 설비 자신의 실적**에만 의존한다(fdc_anomaly의
severity 참고). 그래서 `EQP-CMP01` 에 공정 실적이 하나 늘어도 나머지 일곱
대의 판독값은 한 행도 바뀌지 않는다.

`random` 모듈 전역 함수는 쓰지 않는다. 전역 상태를 공유하므로 호출 순서가
값에 영향을 준다. 항상 `random.Random(seed(...))` 인스턴스를 만들어 쓴다.
"""

from __future__ import annotations

import math
import random
from datetime import datetime, timedelta, timezone


NORMAL = "Normal"
WARNING = "Warning"
ALARM = "Alarm"

RUNNING = "Run"
IDLE = "Idle"

# 유휴 샘플링 간격. 30초의 배수여야 한다. align_to_grid 가 epoch 기준이라
# 배수이기만 하면 유휴 격자가 런 격자의 부분집합이 되어 중복이 없다.
IDLE_INTERVAL_SEC = 300


def align_to_grid(moment: datetime, interval_sec: int = SAMPLE_INTERVAL_SEC) -> datetime:
    """격자 시각으로 내림. 격자는 epoch 기준이라 실행 시각과 무관하다."""
    if moment.tzinfo is None:
        moment = moment.replace(tzinfo=timezone.utc)
    epoch = int(moment.timestamp())
    return datetime.fromtimestamp(epoch - epoch % interval_sec, tz=timezone.utc)


def grid_timestamps(
    start: datetime, end: datetime, interval_sec: int = SAMPLE_INTERVAL_SEC
) -> list[datetime]:
    """start 초과 end 이하의 격자 시각.

    start 를 제외하는 이유는 watermark 를 그대로 넘겨받기 때문이다. 이미
    적재한 마지막 시각을 다시 만들면 중복 행이 생긴다.
    """
    if end < start:
        return []
    first = align_to_grid(start, interval_sec) + timedelta(seconds=interval_sec)
    last = align_to_grid(end, interval_sec)
    out = []
    current = first
    while current <= last:
        out.append(current)
        current += timedelta(seconds=interval_sec)
    return out


def diurnal(sensor: SensorDef, eqp_id: str, moment: datetime) -> float:
    """하루 주기 성분. 24시간 백필 차트에서 눈에 보이는 패턴을 만든다."""
    seconds_of_day = moment.hour * 3600 + moment.minute * 60 + moment.second
    phase = 2 * math.pi * (seed(eqp_id, sensor.sensor_code, "phase") / 2**32)
    return sensor.diurnal_amp * math.sin(2 * math.pi * seconds_of_day / 86400 + phase)


def noise(sensor: SensorDef, eqp_id: str, moment: datetime) -> float:
    """가우시안 잡음. 타임스탬프마다 고정이다."""
    rng = random.Random(seed(eqp_id, sensor.sensor_code, int(moment.timestamp())))
    return rng.gauss(0.0, sensor.sigma)


def excursion_offset(
    sensor: SensorDef, run: Run | None, moment: datetime, profile: EquipmentProfile
) -> float:
    """이상 구간의 이탈량. 런 밖이면 0 이다.

    진폭도 대상 센서도 시점도 전부 MES 불량 실적에서 온다. 그래야
    Eventhouse 에서 찾은 이상이 MES 의 실제 공정이력과 맞아떨어진다.
    """
    if run is None:
        return 0.0
    scale = run_excursion(run, sensor.sensor_code, moment, profile)
    if scale == 0.0:
        return 0.0
    half = (sensor.normal_max - sensor.normal_min) / 2
    return scale * half


def reading_value(
    sensor: SensorDef,
    eqp_id: str,
    moment: datetime,
    profile: EquipmentProfile,
    run: Run | None = None,
) -> float:
    value = (
        sensor.base
        + diurnal(sensor, eqp_id, moment)
        + noise(sensor, eqp_id, moment)
        + excursion_offset(sensor, run, moment, profile)
    )
    return round(value, 4)


def classify(sensor: SensorDef, value: float) -> str:
    if value < sensor.alarm_min or value > sensor.alarm_max:
        return ALARM
    if value < sensor.normal_min or value > sensor.normal_max:
        return WARNING
    return NORMAL


def build_readings(facts, start: datetime, end: datetime) -> list[dict]:
    """구간 안의 모든 판독값.

    30초 격자를 하나만 깔고 각 시각을 런/유휴로 나눈다. 격자를 둘 만들지
    않는 이유는 한 시각이 양쪽에 속해 중복 행이 생기는 것을 막기 위해서다.
    IDLE_INTERVAL_SEC 가 30초의 배수이고 격자가 epoch 기준이라, 유휴 격자는
    런 격자의 부분집합이다.

    런 중에는 센서 6종을 30초마다, 유휴에는 공통 2종을 5분마다 낸다. 멈춘
    설비의 챔버 압력을 30초마다 적는 FDC 는 없다.

    로트 번호는 이상 배치 계산에만 쓰고 행에는 넣지 않는다. 어느 로트였는지는
    MES 에 물어야 한다.
    """
    profiles = build_profiles(facts)
    all_runs = runs_by_equipment(facts)
    moments = grid_timestamps(start, end)
    idle = idle_sensors()
    rows: list[dict] = []

    for profile in profiles.values():
        runs = all_runs.get(profile.eqp_id, [])
        running_sensors = sensors_for(profile.eqp_type)
        for moment in moments:
            run = run_at(runs, moment)
            if run is not None:
                active, run_status = running_sensors, RUNNING
            elif int(moment.timestamp()) % IDLE_INTERVAL_SEC == 0:
                active, run_status = idle, IDLE
            else:
                continue
            for sensor in active:
                value = reading_value(sensor, profile.eqp_id, moment, profile, run)
                rows.append(
                    {
                        "reading_ts": moment,
                        "eqp_id": profile.eqp_id,
                        "eqp_type": profile.eqp_type,
                        "step_code": profile.step_code,
                        "run_status": run_status,
                        "sensor_code": sensor.sensor_code,
                        "value": value,
                        "unit": sensor.unit,
                        "status": classify(sensor, value),
                    }
                )
    return rows


In [ ]:
"""KQL 테이블 스키마와 Spark 적재용 행 변환.

테이블은 Spark 커넥터의 `tableCreateOptions=CreateIfNotExist` 가 만들고,
컬럼 타입은 `spark_schema()` 가 데이터프레임에 명시해 고정한다. 타입 추론에
맡기면 판독값이 우연히 모두 정수인 배치에서 `long` 컬럼이 만들어지고 이후
실수 적재가 조용히 잘린다.

`TABLE_DDL` 과 `RETENTION_DDL` 은 노트북이 **실행하지 않고 출력만** 한다.
Spark 커넥터는 데이터 평면 전용이라 `.create-merge` 같은 제어 명령을 보낼 수
없고, 그걸 보내려면 `azure-kusto-data` 를 따로 설치해야 해서 실습자마다
설치 단계가 하나 늘기 때문이다. 대신 실습자가 KQL 쿼리셋에 붙여넣어 스키마를
확인하거나 다른 작업 영역으로 옮길 때 쓰는 자료로 둔다.
"""

from __future__ import annotations

from datetime import datetime, timezone

SPEC_TABLE = "fdc_sensor_spec"
READING_TABLE = "fdc_sensor_reading"

SPEC_COLUMNS: tuple[str, ...] = (
    "sensor_code",
    "sensor_name_ko",
    "eqp_type",
    "unit",
    "normal_min",
    "normal_max",
    "alarm_min",
    "alarm_max",
    "sample_interval_sec",
    "is_active",
)

READING_COLUMNS: tuple[str, ...] = (
    "reading_ts",
    "eqp_id",
    "eqp_type",
    "step_code",
    "run_status",
    "sensor_code",
    "value",
    "unit",
    "status",
)

# KQL 타입. Spark 데이터프레임 스키마와 이 표가 어긋나면 적재가 조용히 실패한다.
SPEC_SCHEMA: tuple[tuple[str, str], ...] = (
    ("sensor_code", "string"),
    ("sensor_name_ko", "string"),
    ("eqp_type", "string"),
    ("unit", "string"),
    ("normal_min", "real"),
    ("normal_max", "real"),
    ("alarm_min", "real"),
    ("alarm_max", "real"),
    ("sample_interval_sec", "int"),
    ("is_active", "bool"),
)

READING_SCHEMA: tuple[tuple[str, str], ...] = (
    ("reading_ts", "datetime"),
    ("eqp_id", "string"),
    ("eqp_type", "string"),
    ("step_code", "string"),
    ("run_status", "string"),
    ("sensor_code", "string"),
    ("value", "real"),
    ("unit", "string"),
    ("status", "string"),
)


def _create_command(table: str, schema: tuple[tuple[str, str], ...]) -> str:
    """`.create-merge` 를 쓴다. 이미 있으면 컬럼을 합치고 없으면 만든다.

    `.create` 는 기존 테이블을 덮어써 적재한 데이터를 날린다. 실습자가
    노트북을 두 번 돌리는 것만으로 데이터가 사라지면 안 된다.
    """
    columns = ", ".join(f"{name}:{kql_type}" for name, kql_type in schema)
    return f".create-merge table {table} ({columns})"


TABLE_DDL: dict[str, str] = {
    SPEC_TABLE: _create_command(SPEC_TABLE, SPEC_SCHEMA),
    READING_TABLE: _create_command(READING_TABLE, READING_SCHEMA),
}

# 판독 테이블만 보존 기간을 둔다. 스케줄마다 쌓이므로 방치하면 용량을 먹는다.
# 스펙 테이블은 42행 정적이라 보존 정책이 필요 없다.
RETENTION_DDL = (
    f".alter-merge table {READING_TABLE} policy retention "
    'softdelete = 30d recoverability = disabled'
)


def to_iso(moment: datetime) -> str:
    """KQL datetime 리터럴로 안전한 ISO 8601 UTC 문자열."""
    if moment.tzinfo is None:
        moment = moment.replace(tzinfo=timezone.utc)
    return moment.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.%fZ")


def to_rows(records: list[dict], columns: tuple[str, ...]) -> list[tuple]:
    """딕셔너리를 컬럼 순서에 맞춘 튜플로. Spark 데이터프레임 입력이 된다.

    `datetime` 은 그대로 넘긴다. Spark 가 TimestampType 으로 받아 커넥터가
    KQL datetime 으로 변환한다. 문자열로 바꾸면 타입이 어긋난다.
    """
    missing = set(columns) - set(records[0]) if records else set()
    if missing:
        raise KeyError(f"행에 없는 컬럼: {sorted(missing)}")
    return [tuple(record[column] for column in columns) for record in records]


def watermark_query(table: str = READING_TABLE) -> str:
    """마지막으로 적재한 시각. 테이블이 없으면 null 한 행이 온다.

    테이블이 없어도 예외를 내지 않아야 한다. 그래야 '진짜 첫 실행' 과 '조회
    실패' 를 구분할 수 있다. 구분하지 못하면 토큰 만료나 스로틀링 한 번이
    전체 재백필로 이어지고, Eventhouse 는 유니크 제약이 없어 10만 행이 그대로
    중복된다.

    `union isfuzzy=true` 만으로는 안 된다. isfuzzy 는 **여러 레그 중 일부**가
    없을 때만 무시한다. 공식 문서가 명시한다.

        If no resolutions were successful, the query returns an error.

    레그가 실제 테이블 하나뿐이면 첫 실행에 그 하나가 없으므로 "성공한
    해석 0건" 이 되어 쿼리 자체가 실패한다. 그래서 항상 해석되는 빈
    `datatable` 레그를 하나 붙인다. 이 레그가 두 가지를 동시에 해결한다.

    1. 해석 성공이 최소 1건이 되어 fuzzy union 이 성립한다
    2. `reading_ts` 의 타입을 제공하므로 뒤따르는 `max(reading_ts)` 가
       컬럼을 해석할 수 있다 (빈 결과에는 컬럼이 없어 SEM0100 으로 죽는다)
    """
    return (
        f"union isfuzzy=true (datatable(reading_ts:datetime)[]), {table}"
        " | summarize last_ts = max(reading_ts)"
    )


def spec_count_query(table: str = SPEC_TABLE) -> str:
    """스펙 테이블의 행 수. 테이블이 없으면 0 이 온다.

    스펙 적재 여부를 판독 테이블의 watermark 로 판정하면 안 된다. 스펙 쓰기가
    판독 쓰기보다 먼저라, 판독 적재가 실패해 재실행될 때마다 스펙 42행이
    다시 쌓인다. 그러면 스펙과 조인하는 모든 질의가 중복 수만큼 팬아웃된다.

    `watermark_query` 와 같은 이유로 빈 `datatable` 레그를 붙인다. 여기서는
    `count()` 라 컬럼 참조가 없지만, 레그가 하나뿐이면 첫 실행에 쿼리가
    에러를 내는 문제는 똑같다.
    """
    return (
        f"union isfuzzy=true (datatable(sensor_code:string)[]), {table}"
        " | summarize rows = count()"
    )


# KQL 타입에 대응하는 Spark 타입. 데이터프레임 스키마를 명시하는 데 쓴다.
_SPARK_TYPE = {
    "string": "STRING",
    "real": "DOUBLE",
    "int": "INT",
    "bool": "BOOLEAN",
    "datetime": "TIMESTAMP",
}


def spark_schema(table: str) -> str:
    """`spark.createDataFrame(rows, schema=...)` 에 넣을 DDL 문자열.

    스키마를 주지 않으면 Spark 가 값에서 타입을 추론한다. 한 배치의 판독값이
    우연히 모두 정수면 LongType 으로 잡히고, 커넥터가 그대로 KQL `long`
    컬럼을 만들어 이후 실수 적재가 조용히 잘린다. 문자열로 두는 이유는 이
    모듈이 pyspark 없이도 import 되어야 오프라인 테스트가 돌기 때문이다.
    """
    schema = {SPEC_TABLE: SPEC_SCHEMA, READING_TABLE: READING_SCHEMA}[table]
    return ", ".join(f"{name} {_SPARK_TYPE[kql_type]}" for name, kql_type in schema)


In [ ]:
"""적재 전 검증.

Eventhouse 는 append-only 다. 잘못 쓴 행을 지우려면 `.drop extents` 로
익스텐트 단위로 지워야 하고, 같은 익스텐트에 섞인 정상 행까지 날아간다.
그래서 쓰기 전에 검사하고, 치명 항목이 걸리면 아무것도 쓰지 않고 멈춘다.

치명이 아닌 항목은 경고만 낸다. 데이터가 틀린 건 아니지만 실습 소재로서
쓸모가 떨어지는 경우다. 예를 들어 경보가 하나도 없으면 "이상 설비를
찾아라"는 실습을 할 수 없다.
"""

from __future__ import annotations

from collections import Counter
from dataclasses import dataclass
from datetime import datetime


FORBIDDEN_COLUMNS = frozenset(
    {"lot_id", "product_code", "wafer_qty", "defect_code", "judgment", "result", "operator"}
)

MAX_ALARM_RATIO = 0.05


@dataclass(frozen=True)
class Check:
    number: int
    name: str
    fatal: bool
    passed: bool
    detail: str = ""
    skipped: bool = False

    @property
    def mark(self) -> str:
        if self.skipped:
            return "SKIP"
        if self.passed:
            return "OK"
        return "FAIL" if self.fatal else "WARN"


class ValidationFailed(RuntimeError):
    """치명 항목이 걸렸다. 적재하면 안 된다."""


def validate(readings: list[dict], facts, watermark: datetime | None = None) -> list[Check]:
    checks: list[Check] = []
    known_eqp = {e["eqp_id"] for e in facts.equipment}
    spec_keys = {(r["eqp_type"], r["sensor_code"]) for r in build_sensor_spec_rows()}
    columns = set(readings[0]) if readings else set()

    leaked = sorted(FORBIDDEN_COLUMNS & (columns | {n for n, _ in READING_SCHEMA}))
    checks.append(
        Check(
            1,
            "무중복 원칙: 로트 관련 컬럼이 없다",
            True,
            not leaked,
            f"유출된 컬럼: {leaked}" if leaked else "FDC 는 로트를 모른다",
        )
    )

    unknown_eqp = sorted({r["eqp_id"] for r in readings} - known_eqp)
    checks.append(
        Check(
            2,
            "모든 eqp_id 가 MES 설비 목록에 있다",
            True,
            not unknown_eqp,
            f"MES 에 없는 설비: {unknown_eqp}" if unknown_eqp else f"설비 {len(known_eqp)}대",
        )
    )

    unknown_sensor = sorted(
        {(r["eqp_type"], r["sensor_code"]) for r in readings} - spec_keys
    )
    checks.append(
        Check(
            3,
            "모든 sensor_code 가 센서 스펙에 있다",
            True,
            not unknown_sensor,
            f"스펙에 없는 센서: {unknown_sensor}" if unknown_sensor else f"센서 스펙 {len(spec_keys)}종",
        )
    )

    misaligned = [
        r for r in readings if int(r["reading_ts"].timestamp()) % SAMPLE_INTERVAL_SEC
    ]
    checks.append(
        Check(
            4,
            f"reading_ts 가 {SAMPLE_INTERVAL_SEC}초 격자에 정렬돼 있다",
            True,
            not misaligned,
            f"어긋난 행 {len(misaligned)}개" if misaligned else "전부 정렬",
        )
    )

    stale = [r for r in readings if watermark and r["reading_ts"] <= watermark]
    checks.append(
        Check(
            5,
            "생성 구간이 watermark 보다 뒤에 있다",
            True,
            not stale,
            f"이미 적재된 시각의 행 {len(stale)}개" if stale else f"watermark={watermark}",
        )
    )

    # 스펙에 없는 센서는 건너뛴다. 3번 검사가 이미 치명으로 잡았고, 여기서
    # 조회를 시도하면 KeyError 로 죽어 검증 리포트 자체가 나오지 않는다.
    spec_lookup = {
        (r["eqp_type"], r["sensor_code"]): sensor_by_code(r["eqp_type"], r["sensor_code"])
        for r in build_sensor_spec_rows()
    }
    mismatched = [
        r
        for r in readings
        if (r["eqp_type"], r["sensor_code"]) in spec_lookup
        and r["status"] != classify(spec_lookup[(r["eqp_type"], r["sensor_code"])], r["value"])
    ]
    checks.append(
        Check(
            6,
            "status 가 센서 한계와 일치한다",
            False,
            not mismatched,
            f"불일치 {len(mismatched)}행" if mismatched else "전부 일치",
        )
    )

    # 7·8 번은 데이터셋 전체의 성질이지 배치 하나의 성질이 아니다. 첫 백필
    # 이후의 배치는 MES 구간을 지난 유휴만 담고, 유휴에는 이상을 싣지 않으므로
    # 경보가 구조적으로 0 이다. 그대로 평가하면 정상 운영 중인 모든 실행이
    # 경고 2건을 뱉어 진짜 경고가 묻힌다.
    running = [r for r in readings if r.get("run_status") == RUNNING]

    counts = Counter(r["status"] for r in readings)
    ratio = counts[ALARM] / len(readings) if readings else 0.0
    checks.append(
        Check(
            7,
            f"Alarm 비율이 0 초과 {MAX_ALARM_RATIO:.0%} 미만이다",
            False,
            (0 < ratio < MAX_ALARM_RATIO) if running else True,
            (
                f"Alarm {counts[ALARM]}행 / 전체 {len(readings)}행 = {ratio:.2%}"
                if running
                else "가동 행이 없는 배치라 평가하지 않습니다"
            ),
            skipped=not running,
        )
    )

    profiles = build_profiles(facts)
    abnormal = Counter(r["eqp_id"] for r in readings if r["status"] != NORMAL)
    ranked = sorted(profiles.values(), key=lambda p: p.defect_rate)
    worst, best = (ranked[-1].eqp_id, ranked[0].eqp_id) if ranked else ("", "")
    checks.append(
        Check(
            8,
            "불량률 상위 설비의 이상이 하위 설비보다 많다",
            False,
            (abnormal[worst] > abnormal[best]) if running else True,
            (
                f"{worst}={abnormal[worst]}행, {best}={abnormal[best]}행"
                if running
                else "가동 행이 없는 배치라 평가하지 않습니다"
            ),
            skipped=not running,
        )
    )

    bad_status = sorted({r.get("run_status") for r in readings} - {RUNNING, IDLE})
    idle_common = {s.sensor_code for s in idle_sensors()}
    idle_leak = sorted(
        {
            r["sensor_code"]
            for r in readings
            if r.get("run_status") == IDLE and r["sensor_code"] not in idle_common
        }
    )
    checks.append(
        Check(
            9,
            "run_status 가 Run/Idle 뿐이고 유휴에 공정 센서가 없다",
            True,
            not bad_status and not idle_leak,
            (
                f"알 수 없는 상태: {bad_status}" if bad_status
                else f"유휴에 섞인 공정 센서: {idle_leak}" if idle_leak
                else f"가동 {sum(1 for r in readings if r['run_status'] == RUNNING):,}행"
            ),
        )
    )

    return checks


def format_report(checks: list[Check]) -> str:
    lines = ["적재 전 검증", "=" * 60]
    for check in checks:
        flag = " (치명)" if check.fatal else ""
        lines.append(f"[{check.mark:4s}] {check.number}. {check.name}{flag}")
        if check.detail:
            lines.append(f"        {check.detail}")
    failed = [c for c in checks if not c.passed and c.fatal]
    warned = [c for c in checks if not c.passed and not c.fatal]
    skipped = [c for c in checks if c.skipped]
    lines.append("=" * 60)
    tail = f", 건너뜀 {len(skipped)}건" if skipped else ""
    lines.append(f"치명 {len(failed)}건, 경고 {len(warned)}건{tail} / 전체 {len(checks)}건")
    return "\n".join(lines)


def raise_on_fatal(checks: list[Check]) -> None:
    failed = [c for c in checks if not c.passed and c.fatal]
    if failed:
        detail = "; ".join(f"{c.number}. {c.name} — {c.detail}" for c in failed)
        raise ValidationFailed(f"치명 검증 실패로 적재를 중단합니다: {detail}")


In [ ]:
# MES 연결 게이트. 여기서 실패하면 이상 주입의 근거가 없으므로 진행하지 않습니다.
if not MES_API_KEY:
    raise ValueError("MES_API_KEY 를 파라미터 셀에 넣으세요.")
if not KUSTO_URI or not KUSTO_DATABASE:
    raise ValueError("KUSTO_URI 와 KUSTO_DATABASE 를 파라미터 셀에 넣으세요.")

PROBE = MesProbe(MES_BASE_URL, MES_API_KEY)
try:
    FACTS = PROBE.fetch_facts()
except Exception as exc:
    raise RuntimeError(
        "MES 연결에 실패했습니다. 확인할 것: "
        "(1) API 키가 올바른가 (2) Fabric Spark 풀에서 외부 인터넷 아웃바운드가 허용되는가 "
        f"(3) {MES_BASE_URL} 가 응답하는가. 원인: {exc}"
    ) from exc

PROFILES = build_profiles(FACTS)
print(f"공정이력 {len(FACTS.process_results)}건 · 설비 {len(FACTS.equipment)}대")
for _p in sorted(PROFILES.values(), key=lambda p: -p.defect_rate):
    print(
        f"  {_p.eqp_id:12s} {_p.eqp_type:10s} 불량 {_p.defect_runs:2d}/{_p.total_runs:2d}"
        f" = {_p.defect_rate:5.1%}  이상센서: {', '.join(hinted_sensors(_p))}"
    )


In [ ]:
# Kusto 커넥터는 Fabric 런타임에 내장돼 있어 설치가 필요 없습니다.
# 토큰은 이 노트북을 실행하는 신원으로 발급됩니다. 복사해 둘 비밀값이 없습니다.
KUSTO_FORMAT = "com.microsoft.kusto.spark.synapse.datasource"


def kusto_token():
    return mssparkutils.credentials.getToken(KUSTO_URI)


def kusto_read(query):
    return (
        spark.read.format(KUSTO_FORMAT)
        .option("kustoCluster", KUSTO_URI)
        .option("kustoDatabase", KUSTO_DATABASE)
        .option("kustoQuery", query)
        .option("accessToken", kusto_token())
        .load()
    )


def kusto_write(frame, table):
    # Transactional write 는 임시 테이블로 넣고 → 폴링하고 → extent 를 옮깁니다.
    # 그동안 화면에 아무것도 안 나와서 멈춘 것처럼 보입니다. 첫 백필은 10만 행
    # 이라 몇 분 걸립니다. 여기서 "Run all" 을 다시 누르면 두 실행이 같은
    # watermark 를 읽고 같은 행을 두 번 씁니다. Eventhouse 에는 유니크 제약이
    # 없어서 조용히 2배가 됩니다. 그래서 기다리라고 먼저 말해 둡니다.
    print(f"  {table} 에 {frame.count():,}행 쓰는 중입니다. 첫 실행은 몇 분 걸립니다.")
    print("  진행 표시가 없어도 정상입니다. 이 셀을 다시 실행하지 마세요.")
    (
        frame.write.format(KUSTO_FORMAT)
        .option("kustoCluster", KUSTO_URI)
        .option("kustoDatabase", KUSTO_DATABASE)
        .option("kustoTable", table)
        .option("accessToken", kusto_token())
        .option("tableCreateOptions", "CreateIfNotExist")
        # 커넥터는 항상 CSV 로 올립니다. 기본값 NoAdjustment 는 문서 그대로
        # "it does nothing" 이라 매핑을 안 만들고 **위치**로 들어갑니다. 그런데
        # .create-merge 는 새 컬럼을 스키마 끝에 붙일 뿐 재배치하지 않습니다.
        # 그래서 이 모듈의 컬럼 순서가 바뀐 뒤 옛 테이블에 쓰면 값이 옆 컬럼으로
        # 밀려 들어갑니다. 타입이 안 맞으면 null 이 되므로 적재는 "성공" 하고
        # 검사 9개도 통과합니다 — 생성한 배치를 보지 테이블을 안 보기 때문입니다.
        #
        # GenerateDynamicCsvMapping 은 {컬럼 이름, 대상 타입, DataFrame 위치} 로
        # 매핑을 만들어 순서를 무의미하게 만듭니다. FailIfNotMatch 가 아닌 이유는
        # 그쪽 forceAdjustSchema 에 빈 타깃 가드가 없어서 — setCsvMapping 에는
        # 있습니다 — 테이블이 없는 첫 실행에서 빈 스키마와 비교하다 던집니다.
        # 테이블 생성보다 먼저 불립니다.
        .option("adjustSchema", "GenerateDynamicCsvMapping")
        # 커넥터 기본값이지만 명시합니다. 커넥터는 내부 micros 를 이 존의
        # LocalDateTime 으로 바꾼 뒤 offset 없는 문자열로 CSV 에 씁니다
        # (RowCSVWriterUtils.getLocalDateTimeFromTimestampWithZone). Kusto 는
        # offset 없는 datetime 을 UTC 로 읽으므로, 이 값이 UTC 가 아니면 모든
        # reading_ts 가 통째로 밀립니다. 적재는 성공하고 값만 틀립니다.
        # 3차 리뷰에서 고친 watermark **읽기** 경로의 짝입니다. 그쪽은 우리가
        # 고쳤고 이쪽은 커넥터 기본값에 기대고 있었습니다.
        .option("timeZone", "UTC")
        # 커넥터 기본값이긴 하지만 명시합니다. 이 노트북의 중복 방지는 "쓰기는
        # 전부 성공하거나 전부 실패한다" 에 기대고 있습니다. Queued 로 바꾸면
        # 워커 일부만 안착할 수 있는데, 행이 설비별로 묶여 있어서 다음 실행의
        # 전역 max(reading_ts) 가 안 써진 설비를 통째로 건너뜁니다. 그 설비의
        # 그 구간은 영구히 빈 채로 남고, 화면에는 아무 경고도 뜨지 않습니다.
        .option("writeMode", "Transactional")
        .mode("Append")
        .save()
    )


print("테이블은 Spark 커넥터가 만듭니다. 컬럼 타입은 spark_schema() 가 고정합니다.")
for _name, _ddl in TABLE_DDL.items():
    print(f"  {_name:20s} {spark_schema(_name)}")

print()
print("같은 스키마의 KQL DDL 입니다. 실행할 필요는 없고, KQL 쿼리셋에 붙여넣어")
print("스키마를 직접 확인하거나 다른 작업 영역에 옮길 때 씁니다.")
for _ddl in TABLE_DDL.values():
    print(f"  {_ddl}")
print(f"  {RETENTION_DDL}")


In [ ]:
# 어디부터 채울지 정합니다. 테이블이 없으면 첫 실행으로 봅니다.
from datetime import datetime, timedelta, timezone

NOW = datetime.now(timezone.utc)

try:
    _row = kusto_read(watermark_query()).collect()
    WATERMARK = _row[0]["last_ts"] if _row else None
except Exception as exc:
    # 여기서 첫 실행으로 간주하고 넘어가면 안 됩니다. watermark_query 는
    # 테이블이 없어도 예외를 내지 않습니다(항상 해석되는 datatable 레그를
    # 붙여 뒀습니다). 그러므로 여기 온 예외는 토큰 만료·스로틀링·네트워크
    # 같은 일시적 실패입니다. 백필로 떨어지면 이미 적재된 10만 행을 통째로
    # 다시 씁니다. 멈추는 편이 낫습니다.
    raise RuntimeError(
        "watermark 조회에 실패해 중단합니다. 첫 실행으로 간주하면 이미 적재된"
        " 구간을 다시 써서 중복이 쌓입니다(Eventhouse 는 유니크 제약이 없습니다)."
        " KUSTO_URI 와 KUSTO_DATABASE 를 확인하고 다시 실행하세요."
        f" 원인: {exc}"
    ) from exc

if WATERMARK is not None and WATERMARK.tzinfo is None:
    # replace() 로 UTC 라벨만 붙이면 안 됩니다. PySpark 의 TimestampType 은
    # 쓰기와 읽기가 비대칭입니다. 쓸 때는 tz-aware 를 넘기므로 calendar.timegm
    # 을 타서 UTC 로 저장되지만, 읽을 때는 datetime.fromtimestamp(ts) 를 tz
    # 인자 없이 부르므로 **드라이버 OS 의 로컬 시각**이 naive 로 돌아옵니다.
    #
    # 그래서 라벨만 바꾸면 드라이버가 UTC 가 아닐 때 watermark 가 오프셋만큼
    # 통째로 어긋납니다. 서울(+9)이면 미래로 가서 그 구간이 영구히 비고,
    # LA(-7)이면 과거로 가서 매 실행이 수천 행을 중복 적재합니다. 쿼리는
    # 성공하고 값만 틀리기 때문에 앞의 어떤 방어도 이걸 잡지 못합니다.
    #
    # naive 에 astimezone 을 쓰면 시스템 로컬로 해석해 변환하므로
    # fromtimestamp 가 한 일을 정확히 되돌립니다.
    WATERMARK = WATERMARK.astimezone(timezone.utc)

# watermark 는 우리가 과거에 쓴 행에서 나오므로 미래일 수 없습니다. 미래라면
# 드라이버 타임존이나 시계가 어긋난 것입니다. 그냥 두면 START > NOW 가 되어
# 매 실행이 0행을 쓰고 "새로 만들 구간이 없습니다" 만 반복하다가 그 구간을
# 영구히 잃습니다. 조용히 잃느니 여기서 시끄럽게 멈춥니다.
if WATERMARK is not None and WATERMARK > NOW + timedelta(minutes=5):
    raise RuntimeError(
        f"watermark({WATERMARK})가 현재 시각({NOW})보다 미래입니다. "
        "우리가 쓴 행에서 나온 값이므로 있을 수 없습니다. Spark 드라이버의 "
        "타임존이나 시계를 확인하세요. 이대로 두면 그 사이 구간이 영구히 빕니다."
    )

# 첫 실행은 MES 공정이력이 시작하는 시각부터 채웁니다. 벽시계 기준으로 최근
# 몇 시간만 채우면 MES 가 아는 구간과 겹치지 않아서, 센서에서 찾은 이상을
# 공정이력에서 확인할 수 없습니다. 이 노트북의 존재 이유가 사라집니다.
MES_FROM, MES_TO = span(FACTS)

if WATERMARK is None:
    MODE = "backfill"
    START = MES_FROM
else:
    MODE = "live"
    START = WATERMARK

# 구간을 잘라내지 않습니다. 잘라내면 watermark 가 잘린 지점이 아니라 NOW 로
# 가버려서 건너뛴 구간을 다시는 채우지 않습니다. 유휴 구간은 5분 간격 2종이라
# 며칠이 밀려도 수만 행에 그칩니다.
print(f"모드={MODE} · watermark={WATERMARK}")
print(f"MES 공정이력 {MES_FROM} ~ {MES_TO}")
print(f"생성 구간 {START} ~ {NOW}")
if NOW > MES_TO:
    _stale = NOW - MES_TO
    print(f"  MES 배포 후 {_stale.days}일 {_stale.seconds // 3600}시간 지났습니다."
          f" 그 이후 구간은 전부 유휴로 채웁니다.")


In [ ]:
SPEC_ROWS = build_sensor_spec_rows()
READINGS = build_readings(FACTS, START, NOW)

print(f"센서 스펙 {len(SPEC_ROWS)}행")
print(f"판독값 {len(READINGS):,}행")
if READINGS:
    _counts = {}
    for _r in READINGS:
        _counts[_r["status"]] = _counts.get(_r["status"], 0) + 1
    for _k in ("Normal", "Warning", "Alarm"):
        _n = _counts.get(_k, 0)
        print(f"  {_k:8s} {_n:7,d}  {_n / len(READINGS):6.2%}")
else:
    print("새로 만들 구간이 없습니다. 30초 격자가 아직 차지 않았습니다.")


In [ ]:
# 적재 전에 검증합니다. Eventhouse 는 append-only 라 잘못 쓴 행을 지우려면
# 익스텐트 단위로 지워야 하고 같은 익스텐트의 정상 행까지 날아갑니다.
if READINGS:
    RESULTS = validate(READINGS, FACTS, watermark=WATERMARK)
    print(format_report(RESULTS))
    raise_on_fatal(RESULTS)
else:
    print("적재할 행이 없어 검증을 건너뜁니다.")


In [ ]:
# 센서 스펙은 42행 정적입니다. 적재 여부를 판독 테이블의 watermark 로
# 판정하면 안 됩니다. 스펙 쓰기가 판독 쓰기보다 먼저라, 판독 적재가 실패해
# 재실행될 때마다 스펙만 42행씩 쌓입니다. 그러면 스펙과 조인하는 질의가
# 전부 중복 수만큼 팬아웃됩니다. 스펙 테이블 자신의 행 수로 판정합니다.
_spec_rows = kusto_read(spec_count_query()).collect()

# summarize count() 는 테이블이 비어도, 없어도 반드시 한 행을 냅니다. 빈
# 결과가 왔다면 조회 자체가 이상한 것이므로 여기서 멈춥니다. 모르는 채로
# 쓰면 스펙이 42행씩 중복되고, 중복은 조인하는 모든 질의를 조용히 부풀립니다.
if not _spec_rows:
    raise RuntimeError(
        f"{SPEC_TABLE} 행 수를 확인하지 못했습니다. 중복 적재를 피하려고 멈춥니다. "
        "KUSTO_URI 와 KUSTO_DATABASE 를 확인하고 다시 실행하세요."
    )

_spec_present = _spec_rows[0]["rows"]
_spec_expected = len(SPEC_ROWS)

# 0 / 정확히 기대값 / 그 외 를 가릅니다. "0 이 아니면 건너뛴다" 로 뭉뚱그리면
# 부분 적재(20행만 남음)가 영구히 방치됩니다. 그러면 스펙에 없는 센서의 판독
# 행이 inner join 에서 조용히 사라집니다. 중복은 눈에 띄지만 누락은 안 띕니다.
if _spec_present == 0:
    _spec_frame = spark.createDataFrame(
        to_rows(SPEC_ROWS, SPEC_COLUMNS), schema=spark_schema(SPEC_TABLE)
    )
    kusto_write(_spec_frame, SPEC_TABLE)
    print(f"{SPEC_TABLE:20} {_spec_frame.count():7,d}행 적재")
elif _spec_present == _spec_expected:
    print(f"{SPEC_TABLE:20} 건너뜀 (이미 {_spec_present:,}행)")
else:
    raise RuntimeError(
        f"{SPEC_TABLE} 이 {_spec_present:,}행입니다. {_spec_expected}행이어야 합니다.\n"
        f"  {_spec_present:,} < {_spec_expected}  이전 적재가 중간에 끊겼습니다. "
        f"스펙에 없는 센서의 판독 행이 조인에서 조용히 사라집니다.\n"
        f"  {_spec_present:,} > {_spec_expected}  중복 적재됐습니다. "
        f"스펙과 조인하는 질의가 중복 수만큼 부풀어 오릅니다.\n"
        f"\n"
        f"스펙이 {_spec_expected}의 배수로 불어났다면 두 실행이 겹친 것입니다. "
        f"그렇다면 {READING_TABLE} 도 같이 중복됐을 가능성이 높습니다. "
        f"스펙만 지우면 에러는 사라지지만 판독은 계속 2배인 채로 남습니다.\n"
        f"\n"
        f"KQL 쿼리셋에서 판독 중복부터 확인하세요.\n"
        f"  {READING_TABLE}\n"
        f"  | summarize n = count() by reading_ts, eqp_id, sensor_code\n"
        f"  | where n > 1 | count\n"
        f"\n"
        f"0 이 아니면 두 테이블을 모두 지우고 다시 실행하세요.\n"
        f"  .drop table {SPEC_TABLE}\n"
        f"  .drop table {READING_TABLE}\n"
        f"0 이면 스펙만 지우면 됩니다.  .drop table {SPEC_TABLE}"
    )

if READINGS:
    _reading_frame = spark.createDataFrame(
        to_rows(READINGS, READING_COLUMNS), schema=spark_schema(READING_TABLE)
    )
    kusto_write(_reading_frame, READING_TABLE)
    print(f"{READING_TABLE:20} {_reading_frame.count():7,d}행 적재")


## 다음 단계

### 1. 스케줄 걸기

노트북 오른쪽 위 **Run > Schedule** 에서 분 단위 반복을 켭니다. **15분을 권합니다.**

3분으로 두고 싶겠지만 권하지 않습니다. 실행이 주기보다 길어지면 다음 실행이
겹치고, 겹친 두 실행은 **같은 watermark 를 읽어 같은 구간을 두 번 씁니다.**
에러는 나지 않고 검증도 통과합니다. 각 실행이 자기 자신과는 일관되기 때문입니다.
첫 백필은 10만 행이라 Spark 세션 기동만으로도 3분을 넘길 수 있습니다.
자세한 것과 복구 방법은 README 의 "겹치면 중복이 쌓입니다" 를 보세요.

**해상도는 주기와 무관합니다**(가동 30초 · 유휴 5분). 매 실행이 watermark부터
지금까지의 격자를 통째로 채우기 때문에, 15분으로 돌려도 30초 간격 데이터가
나옵니다. 스케줄은 백필이 끝난 것을 확인한 뒤에 켜세요.

### 2. Eventhouse에서 확인하기

데이터는 MES 공정이력 구간(약 65시간)을 덮습니다. 그 이후 시각은 전부 유휴라
`ago(24h)` 로 거르면 가동 구간을 통째로 놓칠 수 있습니다. 먼저 구간을 봅니다.

```kusto
// 데이터가 어느 구간을 덮고 있나?
fdc_sensor_reading
| summarize 시작 = min(reading_ts), 종료 = max(reading_ts), 행 = count() by run_status
```

```kusto
// 경보가 몰린 설비와 시각. 교차 질의의 출발점입니다.
fdc_sensor_reading
| where status == "Alarm" and run_status == "Run"
| summarize 건수 = count(), 시작 = min(reading_ts), 종료 = max(reading_ts)
    by eqp_id, sensor_code
| order by 건수 desc
```

```kusto
// 설비별 주변 온도 추이. 공통 센서라 전 설비를 한 차트에서 비교합니다.
fdc_sensor_reading
| where sensor_code == "AMBIENT_TEMP"
| make-series avg(value) default=0 on reading_ts step 10m by eqp_id
| render timechart
```

```kusto
// 한계치는 판독값에 복사하지 않고 스펙 테이블과 조인해 얻습니다.
fdc_sensor_reading
| where run_status == "Run"
| join kind=inner fdc_sensor_spec on eqp_type, sensor_code
| extend 여유 = normal_max - value
| project reading_ts, eqp_id, sensor_code, value, normal_min, normal_max, 여유
```

### 3. 세 시스템을 함께 봐야 답이 나오는 질문

FDC만으로는 절반까지밖에 못 갑니다. 나머지는 MES와 QMS에 물어야 합니다.

- 경보가 가장 많았던 설비는 어디이고, **그 시각에 그 설비에서
  어떤 로트를 처리하고 있었나요?** (FDC → MES)
- 그 로트들은 품질 검사를 통과했나요? (MES → QMS)
- QMS에서 `Overlay` 결함이 나온 로트를 처리한 설비의 **온도 추이**는 어땠나요?
  (QMS → MES → FDC)
- 센서는 정상 범위였는데 결함이 난 건이 있나요? 설비 문제가 아니라면
  무엇을 봐야 할까요? (FDC + QMS)
- 불량률이 가장 높은 설비의 센서 중 실제로 이탈한 것은 무엇인가요?
  MES 불량코드와 물리적으로 맞아떨어지나요? (MES → FDC)
